In [1]:
import os
import logging
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from dotenv import load_dotenv
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

load_dotenv()

# Setup logging & plotting style
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (10, 5)

# Environment Variables
MINIO_ENDPOINT = os.environ.get("MINIO_API_HOST")
ACCESS_KEY = os.environ.get("ACCESS_KEY")
SECRET_KEY = os.environ.get("SECRET_KEY")
URL_S3_BRONZE = os.environ.get("URL_S3_BRONZE")
URL_S3_SILVER = os.environ.get("URL_S3_SILVER")

# Create PySpark Session
spark = (
    SparkSession.builder.appName("EDA_Notebook_Profiling")
    .config("spark.jars.packages", "org.apache.hadoop:hadoop-aws:3.3.4,com.amazonaws:aws-java-sdk-bundle:1.12.262")
    .config("spark.hadoop.fs.s3a.impl", "org.apache.hadoop.fs.s3a.S3AFileSystem")
    .config("spark.hadoop.fs.s3a.access.key", ACCESS_KEY)
    .config("spark.hadoop.fs.s3a.secret.key", SECRET_KEY)
    .config("spark.hadoop.fs.s3a.endpoint", MINIO_ENDPOINT)
    .config("spark.hadoop.fs.s3a.path.style.access", "true")
    .getOrCreate()
)

print("✅ Spark Session created successfully!")

26/10/07 17:43:02 WARN Utils: Your hostname, minh-HP-250-G8-Notebook-PC resolves to a loopback address: 127.0.1.1; using 192.168.1.15 instead (on interface wlo1)
26/10/07 17:43:02 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address


:: loading settings :: url = jar:file:/home/minh/Documents/Docs/Workspace/Projects/assecing_user_trend_detection/.venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/minh/.ivy2/cache
The jars for the packages stored in: /home/minh/.ivy2/jars
org.apache.hadoop#hadoop-aws added as a dependency
com.amazonaws#aws-java-sdk-bundle added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-c4501597-0a22-4718-b925-089b198c1651;1.0
	confs: [default]
	found org.apache.hadoop#hadoop-aws;3.3.4 in central
	found com.amazonaws#aws-java-sdk-bundle;1.12.262 in central
	found org.wildfly.openssl#wildfly-openssl;1.0.7.Final in central
:: resolution report :: resolve 295ms :: artifacts dl 9ms
	:: modules in use:
	com.amazonaws#aws-java-sdk-bundle;1.12.262 from central in [default]
	org.apache.hadoop#hadoop-aws;3.3.4 from central in [default]
	org.wildfly.openssl#wildfly-openssl;1.0.7.Final from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|

✅ Spark Session created successfully!


In [2]:
print(MINIO_ENDPOINT)

http://localhost:9000


In [3]:
categories = ["All_Beauty", "Amazon_Fashion"]
bronze_stats = []

for cat in categories:
    try:
        bronze_path = f"{URL_S3_BRONZE}/reviews/amazon_reviews_{cat}_2023.jsonl.gz"
        df_bronze = spark.read.json(bronze_path)
        
        total_raw = df_bronze.count()
        distinct_products = df_bronze.select("asin").distinct().count()
        distinct_users = df_bronze.select("user_id").distinct().count()
        null_texts = df_bronze.filter(F.col("text").isNull() | (F.col("text") == "")).count()
        null_ratings = df_bronze.filter(F.col("rating").isNull()).count()
        
        time_range = df_bronze.select(
            F.from_unixtime(F.col("timestamp")/1000).alias("date")
        ).agg(F.min("date").alias("min_date"), F.max("date").alias("max_date")).collect()[0]
        
        bronze_stats.append({
            "Category": cat,
            "Total_Raw_Records": total_raw,
            "Distinct_Products": distinct_products,
            "Distinct_Users": distinct_users,
            "Null_Review_Texts": null_texts,
            "Null_Ratings": null_ratings,
            "Min_Date": time_range["min_date"],
            "Max_Date": time_range["max_date"]
        })
    except Exception as e:
        print(f"⚠️ Cannot read Bronze for category {cat}: {e}")

df_bronze_summary = pd.DataFrame(bronze_stats)
print("=== BRONZE LAYER PROFILING SUMMARY ===")
display(df_bronze_summary)

26/10/07 17:43:08 WARN MetricsConfig: Cannot locate configuration: tried hadoop-metrics2-s3a-file-system.properties,hadoop-metrics2.properties
26/10/07 17:56:15 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://amazon-reviews-datalake/bronze/reviews/amazon_reviews_All_Beauty_2023.jsonl.gz.
org.apache.hadoop.fs.s3a.AWSClientIOException: getFileStatus on s3a://amazon-reviews-datalake/bronze/reviews/amazon_reviews_All_Beauty_2023.jsonl.gz: com.amazonaws.SdkClientException: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:214)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3861)
	at org.apa

⚠️ Cannot read Bronze for category All_Beauty: An error occurred while calling o40.json.
: org.apache.hadoop.fs.s3a.AWSClientIOException: getFileStatus on s3a://amazon-reviews-datalake/bronze/reviews/amazon_reviews_All_Beauty_2023.jsonl.gz: com.amazonaws.SdkClientException: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:214)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3799)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$exists$34(S3AFileSystem.java:4703)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStati

26/10/07 18:00:20 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://amazon-reviews-datalake/bronze/reviews/amazon_reviews_Amazon_Fashion_2023.jsonl.gz.
org.apache.hadoop.fs.s3a.AWSClientIOException: getFileStatus on s3a://amazon-reviews-datalake/bronze/reviews/amazon_reviews_Amazon_Fashion_2023.jsonl.gz: com.amazonaws.SdkClientException: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:214)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3861)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$isDirecto

⚠️ Cannot read Bronze for category Amazon_Fashion: An error occurred while calling o47.json.
: org.apache.hadoop.fs.s3a.AWSClientIOException: getFileStatus on s3a://amazon-reviews-datalake/bronze/reviews/amazon_reviews_Amazon_Fashion_2023.jsonl.gz: com.amazonaws.SdkClientException: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:214)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3799)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$exists$34(S3AFileSystem.java:4703)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5

""


In [4]:
silver_dfs = {}
silver_stats = []

for cat in categories:
    silver_path = f"{URL_S3_SILVER}/reviews/{cat}"
    df_silver = spark.read.parquet(silver_path)
    silver_dfs[cat] = df_silver
    
    total_clean = df_silver.count()
    distinct_products = df_silver.select("parent_asin").distinct().count()
    null_texts = df_silver.filter(F.col("review_text").isNull() | (F.col("review_text") == "")).count()
    
    # Compute Word Count Metrics
    df_silver = df_silver.withColumn("word_count", F.size(F.split(F.col("review_text"), r"\s+")))
    
    word_stats = df_silver.select(
        F.min("word_count").alias("min_w"),
        F.avg("word_count").alias("avg_w"),
        F.max("word_count").alias("max_w"),
        F.percentile_approx("word_count", 0.50).alias("p50_w"),
        F.percentile_approx("word_count", 0.95).alias("p95_w"),
        F.percentile_approx("word_count", 0.99).alias("p99_w")
    ).collect()[0]
    
    silver_stats.append({
        "Category": cat,
        "Clean_Records": total_clean,
        "Distinct_Products": distinct_products,
        "Null_Review_Texts": null_texts,
        "Min_Word_Count": word_stats["min_w"],
        "Avg_Word_Count": round(word_stats["avg_w"], 2),
        "Median_50th": word_stats["p50_w"],
        "Percentile_95th": word_stats["p95_w"],
        "Percentile_99th": word_stats["p99_w"],
        "Max_Word_Count": word_stats["max_w"]
    })

df_silver_summary = pd.DataFrame(silver_stats)
print("=== SILVER LAYER PROFILING SUMMARY ===")
display(df_silver_summary)

26/10/07 18:03:55 WARN FileStreamSink: Assume no metadata directory. Error while looking for metadata directory in the path: s3a://amazon-reviews-datalake/silver/reviews/All_Beauty.
org.apache.hadoop.fs.s3a.AWSClientIOException: getFileStatus on s3a://amazon-reviews-datalake/silver/reviews/All_Beauty: com.amazonaws.SdkClientException: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:214)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3861)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$isDirectory$35(S3AFileSystem.java:4724)
	at org.apache.hadoop.fs.statistics

Py4JJavaError: An error occurred while calling o54.parquet.
: org.apache.hadoop.fs.s3a.AWSClientIOException: getFileStatus on s3a://amazon-reviews-datalake/silver/reviews/All_Beauty: com.amazonaws.SdkClientException: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused: Unable to execute HTTP request: Connect to localhost:9000 [localhost/127.0.0.1] failed: Connection refused
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:214)
	at org.apache.hadoop.fs.s3a.S3AUtils.translateException(S3AUtils.java:175)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.s3GetFileStatus(S3AFileSystem.java:3799)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.innerGetFileStatus(S3AFileSystem.java:3688)
	at org.apache.hadoop.fs.s3a.S3AFileSystem.lambda$exists$34(S3AFileSystem.java:4703)
	at org.apache.hadoop.fs.statistics.impl.IOStatisticsBinding.lambda$trackDurationOfOperation$5(IOStatisticsBinding.java:499)
	at org.apache.hadoop.fs.statistics

In [ ]:
# Combine categories for overall sentiment distribution
combined_df = None
for cat, df in silver_dfs.items():
    if combined_df is None:
        combined_df = df
    else:
        combined_df = combined_df.union(df)

# Map Rating to Sentiment Label (0: Negative, 1: Neutral, 2: Positive)
combined_df = combined_df.withColumn(
    "sentiment_class",
    F.when(F.col("rating").between(1.0, 2.0), "0_Negative (1-2)")
    .when(F.col("rating") == 3.0, "1_Neutral (3)")
    .when(F.col("rating").between(4.0, 5.0), "2_Positive (4-5)")
    .otherwise("Unknown")
)

# Group & calculate percentage
sentiment_dist = combined_df.groupBy("sentiment_class").count().toPandas()
sentiment_dist["percentage"] = round((sentiment_dist["count"] / sentiment_dist["count"].sum()) * 100, 2)
sentiment_dist = sentiment_dist.sort_values("sentiment_class")

print("=== CLASS DISTRIBUTION SUMMARY ===")
display(sentiment_dist)

# Plot Class Imbalance Bar Chart
plt.figure(figsize=(8, 4))
ax = sns.barplot(data=sentiment_dist, x="sentiment_class", y="count", palette="Blues_d")
plt.title("Class Distribution across Sentiment Labels", fontsize=14, fontweight="bold")
plt.xlabel("Sentiment Label", fontsize=12)
plt.ylabel("Number of Reviews", fontsize=12)

for p in ax.patches:
    height = p.get_height()
    pct = (height / sentiment_dist["count"].sum()) * 100
    ax.annotate(f"{height:,.0f}\n({pct:.1f}%)", (p.get_x() + p.get_width() / 2., height / 2),
                ha='center', va='center', color='white', fontweight='bold')

plt.show()

In [ ]:
# EDA Bronze Layer

In [ ]:
# Sample 10% data for fast plotting
sample_texts = combined_df.sample(withReplacement=False, fraction=0.1, seed=42)
sample_texts = sample_texts.withColumn("word_count", F.size(F.split(F.col("review_text"), r"\s+"))).toPandas()

plt.figure(figsize=(10, 4))
sns.histplot(sample_texts[sample_texts["word_count"] <= 250]["word_count"], bins=50, kde=True, color="teal")
plt.title("Distribution of Review Word Counts (Trimmed at 250 words)", fontsize=14, fontweight="bold")
plt.xlabel("Word Count", fontsize=12)
plt.ylabel("Frequency", fontsize=12)
plt.axvline(sample_texts["word_count"].median(), color="red", linestyle="--", label=f'Median: {sample_texts["word_count"].median():.0f} words')
plt.axvline(sample_texts["word_count"].quantile(0.95), color="orange", linestyle="--", label=f'95th Percentile: {sample_texts["word_count"].quantile(0.95):.0f} words')
plt.legend()
plt.show()

In [ ]:
# Audit 1: Check for potential duplicate reviews in Silver layer
# Silver Schema sử dụng 'parent_asin' làm mã sản phẩm
prod_col = "parent_asin" if "parent_asin" in combined_df.columns else "asin"
dup_count = combined_df.groupBy("review_text", prod_col).count().filter(F.col("count") > 1).count()
print(f"🔍 Silver Layer Remaining Duplicates Count: {dup_count}")

# Audit 2: Outlier reviews with extremely long word count (> 1000 words)
outlier_count = combined_df.withColumn("word_count", F.size(F.split(F.col("review_text"), r"\s+"))).filter(F.col("word_count") > 1000).count()
print(f"🔍 Outlier Long Reviews (> 1000 words) Count: {outlier_count}")

# Audit 3: Check rating boundaries
invalid_ratings = combined_df.filter((F.col("rating") < 1.0) | (F.col("rating") > 5.0) | F.col("rating").isNull()).count()
print(f"🔍 Invalid Ratings Count: {invalid_ratings}")